# Лабораторная работа № 3. Логистическая регрессия и оценка качества

**Курс:** Классическое машинное обучение, 4 курс прикладной математики

## Цель работы

Реализовать логистическую регрессию с градиентным спуском, изучить метрики классификации и методы визуализации качества.

**Используемые инструменты:** `numpy`, `matplotlib`, `sklearn.metrics`, `sklearn.datasets`.

### Регламент сдачи

Работа сдаётся в виде этого же ноутбука, дополненного вашим кодом. Обязательно:

1. Читаемый код с комментариями.
2. Визуализации (графики, таблицы).
3. **Текстовый вывод после каждого задания** — не только код, но и объяснение результата.
4. Финальный вывод по работе.

**Критерии оценки:** корректность реализации — 30 %, качество визуализаций и анализа — 20 %,
обоснованность выводов — 20 %, сравнение с эталонными реализациями — 15 %,
оригинальность и дополнительная работа — 15 %.

> Ячейки, помеченные `# TODO`, нужно заполнить самостоятельно.
> Ячейки с готовым кодом можно просто выполнить — они подготавливают данные и графики.

## Подготовка окружения

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
sns.set_palette("viridis")

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

print("train:", X_train_s.shape, " баланс классов:", np.bincount(y_train))

## Задание 1. Класс `LogisticRegressionGD`

Модель: $p(x) = \sigma(w^T x)$, где $\sigma(z) = \dfrac{1}{1 + e^{-z}}$.

Функция потерь — бинарная кросс-энтропия:

$$L(w) = -\frac{1}{n}\sum_{i=1}^{n}\Bigl[y_i \log p_i + (1 - y_i)\log(1 - p_i)\Bigr].$$

Её градиент устроен на удивление просто:

$$\nabla_w L = \frac{1}{n} X^T (p - y).$$

**Численная устойчивость:** при больших $|z|$ прямое вычисление $e^{-z}$ переполняется,
а $\log(0)$ даёт `-inf`. Используйте `np.clip(p, 1e-15, 1 - 1e-15)` в функции потерь.

In [ ]:
class LogisticRegressionGD:
    """Логистическая регрессия, обучаемая полным градиентным спуском."""

    def __init__(self, fit_intercept=True):
        self.fit_intercept = fit_intercept
        self.w = None
        self.loss_history_ = []

    @staticmethod
    def _sigmoid(z):
        # TODO: устойчивая сигмоида (подсказка: scipy.special.expit)
        raise NotImplementedError

    def fit(self, X, y, lr=0.01, epochs=1000):
        # TODO:
        #   1. добавьте столбец единиц, инициализируйте w нулями
        #   2. на каждой эпохе: p = sigmoid(X @ w); grad = X.T @ (p - y) / n; w -= lr * grad
        #   3. сохраняйте значение кросс-энтропии в self.loss_history_
        raise NotImplementedError

    def predict_proba(self, X):
        # TODO
        raise NotImplementedError

    def predict(self, X, threshold=0.5):
        # TODO
        raise NotImplementedError

In [ ]:
# TODO: обучите модель и постройте график сходимости loss_history_.
#       Попробуйте lr = 0.001 / 0.01 / 0.1 / 1.0 — нарисуйте все кривые на одном графике.
#       При каком lr спуск расходится?

## Задание 2. Сравнение с эталоном

Сравните свою реализацию с `sklearn.linear_model.LogisticRegression`
по accuracy и по значениям весов.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# TODO: обучите эталон, сравните accuracy и корреляцию векторов весов.
#       Учтите: sklearn по умолчанию применяет L2-регуляризацию (C=1.0),
#       поэтому для честного сравнения поставьте C очень большим (penalty=None).

## Задание 3. ROC-кривая и AUC

ROC-кривая — зависимость $TPR = \dfrac{TP}{TP+FN}$ от $FPR = \dfrac{FP}{FP+TN}$ при всех порогах.
AUC численно равна вероятности того, что случайный положительный объект получит больший скор,
чем случайный отрицательный.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# TODO: постройте ROC-кривые своей модели и эталона на одном графике,
#       добавьте диагональ случайного классификатора и подпишите AUC в легенде

## Задание 4. Матрица ошибок

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# TODO: постройте матрицу ошибок и выведите classification_report.
#       Прокомментируйте: какая ошибка опаснее в медицинской задаче —
#       пропустить злокачественную опухоль (FN) или перестраховаться (FP)?

## Задание 5. Precision и Recall в зависимости от порога

Постройте график: по оси $x$ — порог от 0 до 1, по оси $y$ — precision, recall и F1.
Найдите порог, максимизирующий F1, и сравните его с порогом 0.5 по умолчанию.

In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score

# TODO: постройте три кривые на одном графике и отметьте оптимальный порог

**Вывод:** *почему порог 0.5 не является универсальным? Как выбирать порог,
если цена ошибок FP и FN различается в 10 раз?*

## Дополнительное задание 1. IRLS (метод Ньютона)

Метод Ньютона для логистической регрессии сводится к итеративно перевзвешенному МНК:

$$w^{(t+1)} = w^{(t)} + (X^T S X)^{-1} X^T (y - p), \qquad S = \operatorname{diag}\bigl(p_i(1 - p_i)\bigr).$$

Сравните число итераций до сходимости с градиентным спуском.

In [ ]:
# TODO: реализуйте IRLS и постройте кривые сходимости обоих методов на одном графике
#       (по оси x — номер итерации, по оси y — значение функции потерь)

## Дополнительное задание 2. $L_2$-регуляризация

Добавьте в свою реализацию штраф $\frac{\lambda}{2}\|w\|^2$ (свободный член не штрафуем).
Градиент получает слагаемое $\lambda w$. Покажите, как λ влияет на качество и на сходимость.

In [ ]:
# TODO: обучите модель для нескольких lambda, постройте график accuracy(lambda) и норму весов ||w||(lambda)

## Финальный вывод

*Напишите здесь связный вывод по работе (5–10 предложений):*

- какие методы вы применили и почему;
- какие результаты получили в числах;
- где реализация «с нуля» разошлась с эталоном из `sklearn` и в чём причина;
- что бы вы улучшили, будь у вас больше времени.